In [1]:
from src.models import GLiNERRagConcat, GLiNERRagCrossAttn
from gliner import GLiNER
import torch
from torchinfo import summary

/Users/jonathanbastinekho/miniconda3/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
text = "Casey watched over the group while Jordan took notes"
context = "Casey is a corrections officer supervising a community service session. Jordan is serving a 6-month sentence for petty theft."
labels = ["officer", "offender", "activity"]

In [2]:
text = "Jojo and albert are in a class."
context = "Jojo is enrolled in Master of Artificial Intelligence. Albert is an associate professor at NTU"
labels = ["student", "teacher", "place"]

In [3]:
model = GLiNERRagConcat("urchade/gliner_large-v1")

entities = model.predict_entities(text, labels, threshold=0.5, context=context)

for entity in entities:
    print(entity["text"], "=>", entity["label"])

/Users/jonathanbastinekho/miniconda3/envs/nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 28777.39it/s]
/Users/jonathanbastinekho/miniconda3/envs/nlp/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model h

Jojo => student
albert => teacher


In [ ]:
base_model = GLiNER.from_pretrained("urchade/gliner_large-v1")

text = """
jojo and albert are in a class. Jojo is enrolled in Master of Artificial Intelligence. Albert is an associate professor at NTU
"""

labels = ["student", "teacher", "place"]

entities = base_model.predict_entities(text, labels)

for entity in entities:
    print(entity["text"], "=>", entity["label"], entity['score'])

/Users/jonathanbastinekho/miniconda3/envs/nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 44384.17it/s]
/Users/jonathanbastinekho/miniconda3/envs/nlp/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model h

jojo => student 0.9914714694023132
albert => student 0.8481825590133667
Jojo => student 0.9884727597236633
Albert => student 0.6958050727844238
associate professor => teacher 0.9480045437812805


# Gliner RAG Cross attn

In [3]:

model = GLiNERRagCrossAttn("urchade/gliner_large-v1")
model.eval()

text = "Jojo and albert are in a class."
context = "Jojo is enrolled in Master of Artificial Intelligence. Albert is an associate professor at NTU"
labels = ["student", "teacher", "place"]

with torch.no_grad():
    entities = model.predict_entities(text, labels, context=context, threshold=0.5)

for e in entities:
    print(e["text"], "=>", e["label"], e['score'])

/Users/jonathanbastinekho/miniconda3/envs/nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 54827.50it/s]
/Users/jonathanbastinekho/miniconda3/envs/nlp/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model h

Jojo => student 1.0
albert => student 1.0
class. => teacher 0.9767928123474121


In [4]:
summary(model)

Layer (type:depth-idx)                                                                Param #
GLiNERRagCrossAttn                                                                    --
├─UniEncoderSpanGLiNER: 1-1                                                           --
│    └─UniEncoderSpanModel: 2-1                                                       --
│    │    └─Encoder: 3-1                                                              (434,438,656)
│    │    └─LstmSeq2SeqEncoder: 3-2                                                   (1,576,960)
│    │    └─SpanRepLayer: 3-3                                                         (7,347,712)
│    │    └─Sequential: 3-4                                                           (2,099,712)
├─ContextCrossAttn: 1-2                                                               --
│    └─MultiheadAttention: 2-2                                                        787,968
│    │    └─NonDynamicallyQuantizableLinear: 3-5              